In [19]:
import tensorflow as tf
import numpy as np
import json, os, shutil

# ── 1. Load ───────────────────────────────────────────────────────────────────
model = tf.keras.models.load_model('shapedetector_model_4b.h5', compile=False)

# ── 2. Strip augmentation layers ──────────────────────────────────────────────
augmentation_layer_types = (
    tf.keras.layers.RandomFlip,
    tf.keras.layers.RandomRotation,
    tf.keras.layers.RandomZoom,
    tf.keras.layers.RandomTranslation,
    tf.keras.layers.RandomContrast,
)

real_layers = [
    l for l in model.layers
    if not isinstance(l, augmentation_layer_types)
    and not (
        hasattr(l, 'layers')
        and all(isinstance(sl, augmentation_layer_types) for sl in l.layers)
    )
]

print("Keeping layers:")
for l in real_layers:
    print(f"  {l.name:30s}  {type(l).__name__}")

# ── 3. Rebuild as clean Sequential ────────────────────────────────────────────
inference_model = tf.keras.Sequential(name="shape_detector")
inference_model.add(tf.keras.Input(shape=(28, 28, 3)))
for layer in real_layers:
    inference_model.add(layer)

inference_model.summary()

# ── 4. Verify ─────────────────────────────────────────────────────────────────
dummy = np.zeros((1, 28, 28, 3), dtype=np.float32)
output = inference_model.predict(dummy)
assert output.shape == (1, 4), f"Unexpected output shape: {output.shape}"
print(f"Output shape  : {output.shape}")
print(f"Output values : {output}")
print(f"Predicted class index: {np.argmax(output)}")

# ── 5. Export as SavedModel and convert to TF.js ──────────────────────────────
for path in ['savedmodel', 'tfjs_model_clean']:
    if os.path.exists(path):
        shutil.rmtree(path)

inference_model.export('savedmodel')

os.system(
    "tensorflowjs_converter "
    "--input_format=tf_saved_model "
    "--output_format=tfjs_graph_model "
    "savedmodel "
    "tfjs_model_clean"
)

os.system("zip -r tfjs_model_clean.zip tfjs_model_clean")

# ── 6. Sanity check ───────────────────────────────────────────────────────────
with open('tfjs_model_clean/model.json') as f:
    m = json.load(f)

print("\nmodel.json top-level keys:", list(m.keys()))
print("Format:", m.get('format', 'unknown'))
print("\nDone. Download tfjs_model_clean.zip")

Keeping layers:
  rescaling_2                     Rescaling
  conv2d_3                        Conv2D
  max_pooling2d_3                 MaxPooling2D
  conv2d_4                        Conv2D
  max_pooling2d_4                 MaxPooling2D
  conv2d_5                        Conv2D
  max_pooling2d_5                 MaxPooling2D
  dropout                         Dropout
  flatten_1                       Flatten
  dense_2                         Dense
  dense_3                         Dense


Model: "shape_detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ (None, 28, 28, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 14, 14, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 7, 7, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 7, 7, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 3, 3, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 3, 3, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 97,956 (382.64 KB)

 Trainable params: 97,956 (382.64 KB)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
Output shape  : (1, 4)
Output values : [[ 0.62451273 -0.37288743 -0.85873675 -0.48490894]]
Predicted class index: 0
Saved artifact at 'savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 3), dtype=tf.float32, name='keras_tensor_360')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137555621364368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137555621364944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137555621364752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137555621365136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137555621363984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137555621367632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137555621369168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137555621367248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137